# Compact IMU 1D CNN: 1,007 parameters

Separate compact model for comparison with the original CNN. Uses the same seven gestures, complete 200-sample trials, stratified 80/20 trial split (seed 42), training-only normalization, and fixed 100 epochs without validation. Participants overlap between splits. Outputs use separate filenames and directories.

Requires numpy, pandas, scikit-learn, and torch. Run all cells in order from a fresh kernel.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

RAW_DIR = Path("/Users/varunnarayanan/Documents/LHNT/IMU Gesture Recognition/raw")  # change to your actual raw folder
OUT_DIR = Path("/Users/varunnarayanan/Documents/LHNT/IMU Gesture Recognition/preprocessed_small_cnn")
SAMPLE_RATE_HZ = 100.0
SAMPLES_PER_TRIAL = 200
TIMESTAMP_TOLERANCE = 0.5  # each interval must be within 50% of 10 ms
CLASS_NAMES = ["forearm_inward", "forearm_lower", "forearm_outward",
               "forearm_raise", "idle", "pronation", "supination"]
N_CLASSES = len(CLASS_NAMES)
LABEL_MAP = {name: i for i, name in enumerate(CLASS_NAMES)}
TRIAL_KEYS = ["participant_id", "session_id", "gesture", "trial_id"]
SENSOR_COLS = ["ax", "ay", "az", "gx", "gy", "gz"]

def load_and_clean(root):
    root = Path(root).expanduser()
    files = sorted(root.glob("P*/S*/*/trial_*.csv"))
    if not files:
        raise FileNotFoundError(f"No trial CSVs found under {root.resolve()}/P*/S*/*/trial_*.csv")
    frames = []
    required = TRIAL_KEYS + ["timestamp_ns", "sequence_number"] + SENSOR_COLS
    for path in files:
        df = pd.read_csv(path, dtype={key: "string" for key in TRIAL_KEYS})
        df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_", regex=False)
        df = df.loc[:, ~df.columns.str.startswith("unnamed")].copy()
        missing = sorted(set(required) - set(df.columns))
        if missing:
            raise ValueError(f"{path}: missing CSV columns {missing}")
        if df.empty:
            raise ValueError(f"{path}: empty trial file")
        for key in TRIAL_KEYS:
            df[key] = df[key].astype("string").str.strip()
        df["gesture"] = df["gesture"].str.lower()
        if df[TRIAL_KEYS].isna().any().any() or (df[TRIAL_KEYS] == "").any().any():
            raise ValueError(f"{path}: missing trial identifiers")
        if (df[TRIAL_KEYS].nunique() != 1).any():
            raise ValueError(f"{path}: a trial file must contain one participant/session/gesture/trial")
        participant, session, gesture, filename = path.relative_to(root).parts
        expected_trial = "T" + path.stem.removeprefix("trial_")
        expected = [participant, session, gesture.lower(), expected_trial]
        actual = df[TRIAL_KEYS].iloc[0].tolist()
        if actual != expected:
            raise ValueError(f"{path}: CSV identifiers {actual} disagree with path {expected}")
        if gesture.lower() not in LABEL_MAP:
            raise ValueError(f"{path}: unknown gesture; update CLASS_NAMES deliberately")
        for col in ["timestamp_ns", "sequence_number"] + SENSOR_COLS:
            df[col] = pd.to_numeric(df[col], errors="coerce")
        numeric = df[["timestamp_ns", "sequence_number"] + SENSOR_COLS].to_numpy(dtype=float)
        if not np.isfinite(numeric).all():
            raise ValueError(f"{path}: missing, nonnumeric, or infinite sensor/timing values")
        for col in ["timestamp_ns", "sequence_number"]:
            if (df[col] % 1 != 0).any():
                raise ValueError(f"{path}: {col} must contain integers")
            df[col] = df[col].astype("int64")
        df["gesture_number"] = LABEL_MAP[gesture.lower()]
        df["source_file"] = str(path)
        frames.append(df)
    combined = pd.concat(frames, ignore_index=True)
    sources = combined.groupby(TRIAL_KEYS)["source_file"].nunique()
    if (sources > 1).any():
        raise ValueError("Multiple files use the same participant/session/gesture/trial identifiers")
    print(f"Loaded {len(files)} trial files from {root.resolve()}")
    return combined

def validate_labels(df):
    missing = set(CLASS_NAMES) - set(df["gesture"])
    if missing:
        raise ValueError(f"Dataset is missing expected classes: {sorted(missing)}")
    print("Class mapping:", LABEL_MAP)
    return CLASS_NAMES.copy()

In [ ]:
def build_trials(df):
    X, labels, records, rejected = [], [], [], []
    expected_dt = 1.0 / SAMPLE_RATE_HZ
    for keys, trial in df.groupby(TRIAL_KEYS, sort=True):
        trial = trial.sort_values("sequence_number")
        seq = trial["sequence_number"].to_numpy()
        # Subtract integer timestamps before converting differences to seconds.
        t = np.diff(trial["timestamp_ns"].to_numpy(dtype=np.int64)) / 1e9
        reasons = []
        if len(trial) != SAMPLES_PER_TRIAL:
            reasons.append(f"{len(trial)} samples, expected {SAMPLES_PER_TRIAL}")
        if len(seq) > 1 and not np.all(np.diff(seq) == 1):
            reasons.append("duplicate sequence numbers or sequence gaps")
        if len(t) and (np.any(t <= 0) or np.any(np.abs(t - expected_dt) > expected_dt * TIMESTAMP_TOLERANCE)):
            reasons.append("irregular timestamps")
        if reasons:
            rejected.append(dict(zip(TRIAL_KEYS, keys)) | {
                "source_file": trial["source_file"].iloc[0], "reason": "; ".join(reasons)})
            continue
        values = trial[SENSOR_COLS].to_numpy(dtype=np.float32)
        if not np.isfinite(values).all():
            raise ValueError(f"{keys}: sensor values overflow float32")
        X.append(values)
        labels.append(int(trial["gesture_number"].iloc[0]))
        records.append(dict(zip(TRIAL_KEYS, keys)) | {
            "y": labels[-1], "start_timestamp_ns": int(trial["timestamp_ns"].iloc[0]),
            "end_timestamp_ns": int(trial["timestamp_ns"].iloc[-1]),
            "n_samples": len(trial), "source_file": trial["source_file"].iloc[0]})
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rejected, columns=TRIAL_KEYS + ["source_file", "reason"]).to_csv(
        OUT_DIR / "rejected_trials.csv", index=False)
    print(f"Accepted {len(X)} trials; rejected {len(rejected)}. See rejected_trials.csv.")
    if not X:
        raise ValueError("No complete trials found. Check sample rate, lengths, and timestamp units.")
    X = np.stack(X)  # (n_trials, 200, 6)
    y = np.array(labels, dtype=np.int64)
    index = pd.DataFrame(records)
    counts = index.groupby(["participant_id", "session_id", "gesture"]).size()
    sessions = index[["participant_id", "session_id"]].drop_duplicates()
    full = pd.MultiIndex.from_tuples(
        [(p, s, g) for p, s in sessions.itertuples(index=False, name=None) for g in CLASS_NAMES],
        names=["participant_id", "session_id", "gesture"])
    counts = counts.reindex(full, fill_value=0)
    counts.rename("valid_trials").to_csv(OUT_DIR / "trial_counts.csv")
    print(counts.unstack("gesture"))
    if (counts != 10).any():
        print("WARNING: some participant/session/gesture groups have other than 10 valid trials.")
    missing = set(range(N_CLASSES)) - set(y.tolist())
    if missing:
        raise ValueError(f"Classes missing after rejection: {[CLASS_NAMES[i] for i in sorted(missing)]}")
    print(f"X: {X.shape}, y: {y.shape}")
    return X, y, index

df = load_and_clean(RAW_DIR)
class_names = validate_labels(df)
X, y, trial_index = build_trials(df)
np.savez_compressed(OUT_DIR / "trials.npz", X=X, y=y,
    participant=trial_index.participant_id.to_numpy(dtype=str),
    session=trial_index.session_id.to_numpy(dtype=str),
    trial=trial_index.trial_id.to_numpy(dtype=str), class_names=np.array(class_names),
    sensor_cols=np.array(SENSOR_COLS), sample_rate_hz=SAMPLE_RATE_HZ)
trial_index.to_csv(OUT_DIR / "trials_index.csv", index_label="input_id")

## Randomized 80/20 trial split

Use random seed 42 and stratify by gesture: with 350 trials, train on 280 (40 per gesture) and test on 70 (10 per gesture). The split balances gestures across the whole dataset, not within each participant. Each complete trial stays in a single set, but participants and sessions may occur in both sets. This evaluates unseen trials from familiar participants; it does not measure unseen-person generalization. Normalization uses training trials only.

In [ ]:
from sklearn.model_selection import train_test_split

groups = trial_index["participant_id"].to_numpy()
SPLIT_SEED = 42
train_idx, test_idx = train_test_split(
    np.arange(len(X)), test_size=0.20, random_state=SPLIT_SEED,
    shuffle=True, stratify=y)
assert not (set(train_idx) & set(test_idx))
assert len(train_idx) + len(test_idx) == len(X)

# A trial is identified by participant, session, gesture, and trial ID.
if trial_index.duplicated(TRIAL_KEYS).any():
    raise ValueError("Duplicate trial identifiers found; cannot safely split")
train_keys = set(trial_index.iloc[train_idx][TRIAL_KEYS].itertuples(index=False, name=None))
test_keys = set(trial_index.iloc[test_idx][TRIAL_KEYS].itertuples(index=False, name=None))
assert train_keys.isdisjoint(test_keys), "A trial occurs in both sets"
train_files = set(trial_index.iloc[train_idx]["source_file"])
test_files = set(trial_index.iloc[test_idx]["source_file"])
assert train_files.isdisjoint(test_files), "A CSV file occurs in both sets"

# Also reject identical sensor recordings saved under different identifiers.
import hashlib
def recording_hash(trial):
    return hashlib.sha256(np.ascontiguousarray(trial).tobytes()).digest()
train_hashes = {recording_hash(X[i]) for i in train_idx}
test_hashes = {recording_hash(X[i]) for i in test_idx}
if not train_hashes.isdisjoint(test_hashes):
    raise ValueError("Identical sensor recordings occur across training and test; inspect duplicate files")
print("Split checks passed: no shared trials, source files, or identical sensor recordings.")
for name, idx in [("train", train_idx), ("test", test_idx)]:
    print(name, "participants:", sorted(set(groups[idx])))
mean = X[train_idx].mean(axis=(0, 1), keepdims=True)
std = X[train_idx].std(axis=(0, 1), keepdims=True).clip(min=1e-6)
X_train = ((X[train_idx] - mean) / std).transpose(0, 2, 1)
X_test = ((X[test_idx] - mean) / std).transpose(0, 2, 1)
y_train, y_test = y[train_idx], y[test_idx]
print("Training / test shapes:", X_train.shape, X_test.shape)
for name, idx in [("train", train_idx), ("test", test_idx)]:
    print(name, "class counts:", np.bincount(y[idx], minlength=N_CLASSES))
split_index = trial_index.copy()
split_index["split"] = "train"
split_index.loc[test_idx, "split"] = "test"
split_index.to_csv(OUT_DIR / "split_index.csv", index_label="input_id")
# Show how many complete trials from each participant enter each split.
participant_split_counts = pd.crosstab(
    split_index["participant_id"], split_index["split"]
).reindex(columns=["train", "test"], fill_value=0)
participant_split_counts["total"] = participant_split_counts.sum(axis=1)
print("\nTrial allocation by participant:")
print(participant_split_counts.to_string())
participant_split_counts.to_csv(OUT_DIR / "participant_split_counts.csv")

# List the source CSV for every held-out test trial.
test_trial_sources = split_index.loc[
    split_index["split"] == "test",
    ["participant_id", "session_id", "gesture", "trial_id", "source_file"]
].copy()
test_trial_sources.index.name = "input_id"
test_trial_sources = test_trial_sources.sort_values(TRIAL_KEYS)
print("\nSource CSVs for every test trial:")
print(test_trial_sources.to_string(index=True))
test_trial_sources.to_csv(OUT_DIR / "test_trial_sources.csv", index=True)


## Compact CNN architecture

Input `(N, 6, 200)` → Conv1D 6→8, kernel 3, padding 1 → ReLU → Conv1D 8→16, kernel 3, padding 1 → ReLU → adaptive average pooling to 4 temporal bins → flatten to 64 features → Linear 64→7.

Four bins retain coarse early-to-late information. The model has 1,007 learned parameters (4,028 bytes of float32 parameter values, excluding metadata). Checkpoint size and Raspberry Pi inference latency must be measured separately.

In [ ]:
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

class SmallIMUCNN(nn.Module):
    def __init__(self, n_channels=6, n_classes=N_CLASSES):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(n_channels, 8, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv1d(8, 16, kernel_size=3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool1d(4),
        )
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(16 * 4, n_classes))

    def forward(self, x):
        return self.classifier(self.features(x))

def loader(x, labels, shuffle=False):
    ds = TensorDataset(torch.from_numpy(x).float(), torch.from_numpy(labels).long())
    return DataLoader(ds, batch_size=32, shuffle=shuffle)

train_loader = loader(X_train, y_train, True)
test_loader = loader(X_test, y_test)
torch.manual_seed(42)
model = SmallIMUCNN(n_channels=len(SENSOR_COLS), n_classes=N_CLASSES)
print(model)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))
assert sum(p.numel() for p in model.parameters()) == 1007
assert model(torch.zeros(2, len(SENSOR_COLS), SAMPLES_PER_TRIAL)).shape == (2, N_CLASSES)

In [ ]:
# Fixed training duration; no validation or early stopping.
torch.manual_seed(42)
N_EPOCHS = 100
torch.set_num_threads(2)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

def evaluate(data_loader):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in data_loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            total_loss += criterion(logits, yb).item() * len(yb)
            correct += (logits.argmax(dim=1) == yb).sum().item()
            total += len(yb)
    return total_loss / total, correct / total

for epoch in range(N_EPOCHS):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        if not torch.isfinite(loss):
            raise RuntimeError("Nonfinite training loss")
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(yb)
        correct += (logits.argmax(dim=1) == yb).sum().item()
        total += len(yb)
    if epoch % 10 == 0 or epoch == N_EPOCHS - 1:
        print(f"Epoch {epoch+1}: training loss={total_loss/total:.4f}, accuracy={correct/total:.3f}")

test_loss, test_accuracy = evaluate(test_loader)
print("Held-out test loss and accuracy:", (test_loss, test_accuracy))
import json
(OUT_DIR / "metrics.json").write_text(json.dumps({
    "test_loss": test_loss, "test_accuracy": test_accuracy,
    "train_trials": len(train_idx), "test_trials": len(test_idx),
    "epochs": N_EPOCHS, "split_seed": 42, "split_unit": "trial",
    "train_participants": sorted(set(groups[train_idx])),
    "test_participants": sorted(set(groups[test_idx]))}, indent=2))
torch.save({"model_state": model.cpu().state_dict(),
            "mean": mean, "std": std, "class_names": class_names,
            "sample_rate_hz": SAMPLE_RATE_HZ, "sensor_cols": SENSOR_COLS,
            "samples_per_trial": SAMPLES_PER_TRIAL, "n_classes": N_CLASSES,
            "architecture": "SmallIMUCNN", "pool_bins": 4,
    "conv_channels": [8, 16], "label_map": LABEL_MAP}, OUT_DIR / "imu_small_cnn.pt")

## Test accuracy per gesture
For each true gesture, reports how many test trials were classified correctly (class recall). Run the cell below after training; no retraining is needed.


In [ ]:
# Run after the training cell; evaluates the current trained model without retraining.
# The saved model is moved to CPU at the end of training, so use its current device.
eval_device = next(model.parameters()).device
model.eval()
class_correct = np.zeros(N_CLASSES, dtype=np.int64)
class_total = np.zeros(N_CLASSES, dtype=np.int64)
with torch.no_grad():
    for xb, yb in test_loader:
        predictions = model(xb.to(eval_device)).argmax(dim=1).cpu().numpy()
        targets = yb.cpu().numpy()
        class_total += np.bincount(targets, minlength=N_CLASSES)
        class_correct += np.bincount(targets[predictions == targets], minlength=N_CLASSES)
accuracy = np.divide(class_correct, class_total,
                     out=np.full(N_CLASSES, np.nan), where=class_total > 0)
per_class_accuracy = pd.DataFrame({
    "gesture": class_names,
    "correct": class_correct,
    "total": class_total,
    "accuracy_percent": accuracy * 100,
})
print(per_class_accuracy.to_string(index=False, float_format=lambda value: f"{value:.1f}"))
print(f"Overall test accuracy: {class_correct.sum()}/{class_total.sum()} "
      f"({100 * class_correct.sum() / class_total.sum():.1f}%)")
per_class_accuracy.to_csv(OUT_DIR / "test_accuracy_per_class.csv", index=False)


## Save compact trained model beside the notebook

Run after training to save `imu_small_cnn.pt` beside this notebook, separately from the original checkpoint.

In [ ]:
# Save the currently trained model beside this notebook; no retraining needed.
# Run training first if the kernel has been restarted.
save_dir = Path("/Users/varunnarayanan/Documents/LHNT/IMU Gesture Recognition")
save_path = save_dir / "imu_small_cnn.pt"
torch.save({
    "model_state": {key: value.detach().cpu().clone()
                    for key, value in model.state_dict().items()},
    "mean": mean,
    "std": std,
    "class_names": class_names,
    "sensor_cols": SENSOR_COLS,
    "sample_rate_hz": SAMPLE_RATE_HZ,
    "samples_per_trial": SAMPLES_PER_TRIAL,
    "n_classes": N_CLASSES,
    "architecture": "SmallIMUCNN", "pool_bins": 4,
    "conv_channels": [8, 16], "label_map": LABEL_MAP,
}, save_path)
print(f"Saved model to: {save_path}")